In [1]:
from typing import Callable

import pandas as pd
import torch
from numpy.ma.extras import vstack
from pandas.core.common import random_state
from path import  Path
from collections import defaultdict, Counter
import re
import numpy as np

In [2]:
N_CLASSES = 4

Для начала попытаемся сопоставить каждой новости два вектора: для заголовка и для описания

сделаем это при помощи TF-IDF

In [3]:
data_dir = Path('../data')

In [4]:
train = pd.read_csv(data_dir / 'train.csv')
test = pd.read_csv(data_dir / 'test.csv')

In [5]:
def wold_level_tokenizer(text):
    r = re.compile(r'(\w+|[^\w\s])')
    return r.findall(text)

In [6]:
from scipy.sparse import csr_matrix, vstack

class TF_IDF:
    def __init__(self, lst : list[str], tokenizer : Callable[[str], list[str]]):
        self.dict = Counter()
        self.tokenizer = tokenizer
        for s in lst:
            self.dict.update(set(self.tokenizer(s)))
        self.token_to_idx = {token: i for i, token in enumerate(self.dict)}
        self.D = len(self.dict)

    def get_tf_idf(self, text : str):
        text_counter = Counter()
        tokens = self.tokenizer(text)
        n_text = len(tokens)
        text_counter.update(tokens)

        shape = (1, self.D)

        vals = []
        inds = []

        for token, count in text_counter.items():
            idx = self.token_to_idx.get(token, -1)
            if idx == -1:
                continue
            inds.append(idx)
            vals.append(count / n_text * (np.log(self.D) - np.log(self.dict[token])))

        emb = csr_matrix(
            (vals, ([0] * len(inds), inds)),
            shape=shape
        )


        return emb

In [7]:
train_list_of_texts = train['Title'].tolist() + train['Description'].tolist()

tf_idf = TF_IDF(
    train_list_of_texts,
    wold_level_tokenizer
)

In [8]:
def make_features_tf_idf(df, tf_idf):
    s = pd.Series(df['Title'] + ' ' + df['Description'])
    return vstack([tf_idf.get_tf_idf(st) for st in s.tolist()], format='csr'), df['Class Index'] - 1

In [9]:
X_train, y_train = make_features_tf_idf(train, tf_idf)

In [10]:
X_test, y_test = make_features_tf_idf(test, tf_idf)

In [11]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

logreg = LogisticRegression(max_iter=1000)

logreg.fit(X_train, y_train)

,"max_iter max_iter: int, default=100Maximum number of iterations taken for the solvers to converge.",1000
,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add an L2 penalty term and it is the default choice;- `'l1'`: add an L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` and `C` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'`, `l1_ratio` set to any float between 0 and 1 for `penalty='elasticnet'`, and `C=np.inf` for `penalty=None`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation <regularized-logistic-loss>`) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneOnly used for `solver` == 'sag', 'saga' or 'liblinear' to shuffle thedata. It has no effect on the other solvers.See :term:`Glossary <random_state>` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use i

In [12]:
def evaluate_accuracy(model, X, y):
    y_pred = model.predict(X)
    return accuracy_score(y, y_pred)

In [13]:
class log:
    def __init__(self):
        self.log = dict()

    def add_log(self, train_acc, test_acc, tokenizer, model, note=''):
        hashh = tokenizer + model
        self.log[hashh] = {
            'Модель' : model,
            'Токенизатор' : tokenizer,
            'train acc' : train_acc,
            'test acc' : test_acc,
            'замечания' : note
        }

    def display(self):
        display(pd.DataFrame(list(self.log.values())).round(3))

In [14]:
LOG = log()

In [15]:
# def compare_acc(model):
#     print(f'acc на train {evaluate_accuracy(model, X_train, y_train)}, на test {evaluate_accuracy(model, X_test, y_test)}')

In [16]:
def add_log(model, model_name, tokenizer, X_train, y_train, X_test, y_test, note='', display=True):
    LOG.add_log(
        evaluate_accuracy(model, X_train, y_train),
        evaluate_accuracy(model, X_test, y_test),
        tokenizer,
        model_name,
        note
    )
    if display:
        LOG.display()

In [17]:
data = [X_train, y_train, X_test, y_test]

In [18]:
add_log(logreg, 'TF-IDF + логрег', 'word-level', *data)

,Модель,Токенизатор,train acc,test acc,замечания
0,TF-IDF + логрег,word-level,0.94,0.918,


In [19]:
from sklearn.dummy import DummyClassifier

dummy = DummyClassifier()
dummy.fit(X_train, y_train)

,"strategy strategy: {""most_frequent"", ""prior"", ""stratified"", ""uniform"", ""constant""}, default=""prior""Strategy to use to generate predictions.* ""most_frequent"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit`. The `predict_proba` method returns the matching one-hot encoded vector.* ""prior"": the `predict` method always returns the most frequent class label in the observed `y` argument passed to `fit` (like ""most_frequent""). ``predict_proba`` always returns the empirical class distribution of `y` also known as the empirical class prior distribution.* ""stratified"": the `predict_proba` method randomly samples one-hot vectors from a multinomial distribution parametrized by the empirical class prior probabilities. The `predict` method returns the class label which got probability one in the one-hot vector of `predict_proba`. Each sampled row of both methods is therefore independent and identically distributed.* ""uniform"": generates predictions uniformly at random from the list of unique classes observed in `y`, i.e. each class has equal probability.* ""constant"": always predicts a constant label that is provided by the user. This is useful for metrics that evaluate a non-majority class. .. versionchanged:: 0.24 The default value of `strategy` has changed to ""prior"" in version 0.24.",'prior'
,"random_state random_state: int, RandomState instance or None, default=NoneControls the randomness to generate the predictions when``strategy='stratified'`` or ``strategy='uniform'``.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",None
,"constant constant: int or str or array-like of shape (n_outputs,), default=NoneThe explicit constant as predicted by the ""constant"" strategy. Thisparameter is useful only for the ""constant"" strategy.",None
Name,Type,Value
"class_prior_ class_prior_: ndarray of shape (n_classes,) or list of such arraysFrequency of each class observed in `y`. For multioutput classificationproblems, this is computed independently for each output.","ndarray[float64](4,)","[0.25,0.25,0.25,0.25]"
"classes_ classes_: ndarray of shape (n_classes,) or list of such arraysUnique class labels observed in `y`. For multi-output classificationproblems, this attribute is a list of arrays as each output has anindependent set of possible classes.","ndarray[int64](4,)","[0,1,2,3]"
n_classes_ n_classes_: int or list of intNumber of label for each output.,int,4
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`.,int,88127
n_outputs_ n_outputs_: intNumber of outputs.,int,1
sparse_output_ sparse_output_: boolTrue if the array returned from predict is to be in sparse CSC format.Is automatically set to True if the input `y` is passed in sparseformat.,bool,False


In [20]:
add_log(dummy, 'самый частый класс', 'нет', *data)

,Модель,Токенизатор,train acc,test acc,замечания
0,TF-IDF + логрег,word-level,0.94,0.918,
1,самый частый класс,нет,0.25,0.250,


уже самая простая модель дала довольно хороший результат на трейне и тесте, попробуем заменить наш токенизатор на токенизатор из библиотеки nltk

In [21]:
from nltk.tokenize import word_tokenize
import nltk
# nltk.download('punkt_tab')

tf_idf_nltk = TF_IDF(
    train_list_of_texts,
    word_tokenize
)

X_train, y_train = make_features_tf_idf(train, tf_idf_nltk)
X_test, y_test = make_features_tf_idf(test, tf_idf_nltk)

data = [X_train, y_train, X_test, y_test]

In [22]:
logreg_nltk = LogisticRegression(max_iter=1000)
logreg_nltk.fit(X_train, y_train)
add_log(logreg_nltk, 'TF-IDF + логрег', 'nltk' , *data)

,Модель,Токенизатор,train acc,test acc,замечания
0,TF-IDF + логрег,word-level,0.940,0.918,
1,самый частый класс,нет,0.250,0.250,
2,TF-IDF + логрег,nltk,0.945,0.918,


Моделть не прибавила в обобщающей способности

Теперь сравним RNN GRU и LSTM

Теперь нам нужен другой класс токенизатора

In [23]:
from torch import nn
from torch.optim import Adam
from torch.utils.data import DataLoader, TensorDataset
import torch
from torch.nn.utils.rnn import pad_sequence
from torchmetrics.classification import MulticlassAccuracy
from torchmetrics.aggregation import MeanMetric
import copy

In [24]:
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

In [25]:
unk_token = '[UNK]'
sep_token = '[SEP]'
pad_token = '[PAD]'

tokenizer = Tokenizer(BPE(unk_token=unk_token))
tokenizer.pre_tokenizer = Whitespace()

trainer = BpeTrainer(special_tokens=[unk_token, sep_token, pad_token], show_progress=False)
tokenizer.train_from_iterator(train_list_of_texts, trainer)

In [26]:
def make_token_seqs(df, tokenizer):
    s = pd.Series(df['Title'] + sep_token + df['Description'])
    y = torch.tensor(df['Class Index'] - 1)

    X = [
        torch.tensor(tokenizer.encode(text).ids, dtype=torch.long)
        for text in s
    ]

    return X, y

In [27]:
X_train, y_train = make_token_seqs(train, tokenizer)
X_test, y_test = make_token_seqs(test, tokenizer)

In [32]:
from tqdm.auto import tqdm

In [36]:
def train_loop(model_class, model_param=None, optimizer_param=None, lr=1e-3, num_epoch=10, batch_size=64,optimizer_class=Adam, criterion=nn.CrossEntropyLoss(), X_train=X_train, y_train=y_train, X_test=X_test, y_test=y_test, eps=1e-9, device='mps', patience=4):
    model_param =  {} if model_param is None else model_param
    optimizer_param =  {} if optimizer_param is None else optimizer_param

    pad_id = tokenizer.token_to_id(pad_token)
    def collate_fn(batch):
        X, y = zip(*batch)
        X = pad_sequence(X, padding_value=pad_id, batch_first=True)
        y = torch.stack(y)
        return X, y

    train_loader = DataLoader(
        list(zip(X_train, y_train)),
        batch_size=batch_size,
        collate_fn=collate_fn,
        shuffle=True
    )

    test_loader = DataLoader(
        list(zip(X_test, y_test)),
        batch_size=batch_size,
        collate_fn=collate_fn
    )

    model = model_class(**model_param).to(device)
    optimizer = optimizer_class(model.parameters(), lr=lr, **optimizer_param)

    accuracy_acc = MulticlassAccuracy(
        num_classes=N_CLASSES,
        average='micro'
    ).to(device)

    locc_acc = MeanMetric().to(device)

    history = {
        'train_acc' : [],
        'train_loss' : [],
        'test_acc' : [],
        'test_loss' : []
    }

    def train_epoch(epoch):
        accuracy_acc.reset()
        locc_acc.reset()
        model.train()

        for X, y in train_loader:
            X = X.to(device)
            y = y.to(device)

            optimizer.zero_grad()
            logits = model(X)
            loss = criterion(logits, y)
            loss.backward()
            optimizer.step()

            with torch.no_grad():
                y_pred = torch.argmax(logits, dim=1)
                accuracy_acc.update(y_pred, y)
                locc_acc.update(loss, weight=X.shape[0])

        return accuracy_acc.compute().item(), locc_acc.compute().item()

    @torch.no_grad
    def test_epoch(epoch):
        accuracy_acc.reset()
        locc_acc.reset()
        model.eval()

        for X, y in test_loader:
            X = X.to(device)
            y = y.to(device)

            logits = model(X)
            loss = criterion(logits, y)

            y_pred = torch.argmax(logits, dim=1)
            accuracy_acc.update(y_pred, y)
            locc_acc.update(loss, weight=X.shape[0])

        return accuracy_acc.compute().item(), locc_acc.compute().item()

    best_loss = float('inf')
    bad_epochs = 0
    best_state = None

    for epoch in tqdm(range(num_epoch), desc="Epochs"):
        train_acc, train_loss = train_epoch(epoch)
        test_acc, test_loss = test_epoch(epoch)
        tqdm.write(
            f' Эпоха {epoch + 1}/{num_epoch}, train_acc {train_acc}, train_loss {train_loss}, test_acc {test_acc}, test_loss {test_loss}'
        )

        history['train_acc'].append(train_acc)
        history['train_loss'].append(train_loss)
        history['test_acc'].append(test_acc)
        history['test_loss'].append(test_loss)

        if test_loss < best_loss - eps:
            best_loss = test_loss
            best_state = copy.deepcopy(model.state_dict())
            bad_epochs = 0
        else:
            bad_epochs += 1

        if bad_epochs >= patience:
            break

    model.load_state_dict(best_state)

    return model, history

In [37]:
def make_rnn_class(arc : nn.Module):
    class RNN(nn.Module):
        def __init__(self, num_layers=1, embedding_dim=128, hidden_dim=64):
            super().__init__()

            self.emb = nn.Embedding(
                num_embeddings=tokenizer.get_vocab_size(),
                embedding_dim=embedding_dim
            )

            self.rnn = arc(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True
            )

            self.fc = nn.Linear(
                in_features=hidden_dim,
                out_features=N_CLASSES
            )

        def forward(self, batch):
            _, state = self.rnn(self.emb(batch))
            h_n = state[0] if isinstance(state, tuple) else state
            return self.fc(h_n[-1])
    return RNN

Тренируем RRN

In [39]:
trained_rnn, history_rnn = train_loop(make_rnn_class(nn.RNN))

Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

 Эпоха 1/10, train_acc 0.25573334097862244, train_loss 1.3839778900146484, test_acc 0.2530263066291809, test_loss 1.3835324048995972
 Эпоха 2/10, train_acc 0.25636667013168335, train_loss 1.3824890851974487, test_acc 0.257763147354126, test_loss 1.3808307647705078
 Эпоха 3/10, train_acc 0.2550416588783264, train_loss 1.3807021379470825, test_acc 0.25381579995155334, test_loss 1.3834097385406494
 Эпоха 4/10, train_acc 0.2575916647911072, train_loss 1.3796429634094238, test_acc 0.2544736862182617, test_loss 1.3788683414459229
 Эпоха 5/10, train_acc 0.26095834374427795, train_loss 1.3780322074890137, test_acc 0.2540789544582367, test_loss 1.3847013711929321


KeyboardInterrupt: 

In [40]:
trained_gru, history_gru = train_loop(make_rnn_class(nn.GRU))

Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

 Эпоха 1/10, train_acc 0.7137083411216736, train_loss 0.6779246926307678, test_acc 0.8822368383407593, test_loss 0.3342861831188202
 Эпоха 2/10, train_acc 0.9116083383560181, train_loss 0.26239466667175293, test_acc 0.906973659992218, test_loss 0.2719756066799164
 Эпоха 3/10, train_acc 0.9388333559036255, train_loss 0.18561193346977234, test_acc 0.902368426322937, test_loss 0.29123011231422424
 Эпоха 4/10, train_acc 0.9557916522026062, train_loss 0.1361016035079956, test_acc 0.9107894897460938, test_loss 0.2742471992969513
 Эпоха 5/10, train_acc 0.9688500165939331, train_loss 0.09908035397529602, test_acc 0.9097368717193604, test_loss 0.2971159517765045
 Эпоха 6/10, train_acc 0.9770333170890808, train_loss 0.07365333288908005, test_acc 0.9103947281837463, test_loss 0.3276348412036896


In [41]:
trained_lstm, history_lstm = train_loop(make_rnn_class(nn.LSTM))

Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

 Эпоха 1/10, train_acc 0.5241833329200745, train_loss 0.9782706499099731, test_acc 0.8222368359565735, test_loss 0.5105670690536499
 Эпоха 2/10, train_acc 0.8799083232879639, train_loss 0.3598250448703766, test_acc 0.8853947520256042, test_loss 0.33205726742744446
 Эпоха 3/10, train_acc 0.9241833090782166, train_loss 0.23312823474407196, test_acc 0.9005263447761536, test_loss 0.2912726402282715
 Эпоха 4/10, train_acc 0.9449416399002075, train_loss 0.17426049709320068, test_acc 0.9034210443496704, test_loss 0.28934526443481445
 Эпоха 5/10, train_acc 0.960141658782959, train_loss 0.13044089078903198, test_acc 0.9081578850746155, test_loss 0.2901470363140106
 Эпоха 6/10, train_acc 0.9712833166122437, train_loss 0.09707598388195038, test_acc 0.902368426322937, test_loss 0.3337039351463318
 Эпоха 7/10, train_acc 0.9797666668891907, train_loss 0.0711749792098999, test_acc 0.906973659992218, test_loss 0.3507382273674011
 Эпоха 8/10, train_acc 0.9847333431243896, train_loss 0.05430315062403679

In [56]:
LOG.add_log(0.91, 0.906,'BPE', 'GRU', 'early stop 5 эпох')
LOG.add_log(0.944, 0.903,'BPE', 'LSTM', 'early stop 7 эпох')
LOG.add_log(0.25, 0.25,'BPE', 'RNN', 'не обучалась, видимо не текли градиенты')
LOG.display()

,Модель,Токенизатор,train acc,test acc,замечания
0,TF-IDF + логрег,word-level,0.940,0.918,
1,самый частый класс,нет,0.250,0.250,
2,TF-IDF + логрег,nltk,0.945,0.918,
3,GRU,BPE,0.910,0.906,early stop 5 эпох
4,LSTM,BPE,0.944,0.903,early stop 7 эпох
5,RNN,BPE,0.250,0.250,"не обучалась, видимо не текли градиенты"
6,GRU_tuned,BPE,0.937,0.910,"32 emb и hidden, 3 слоя"


В данном сеттинге мы не смогли превзойти самую первую и простую модель. Так как датасет у нас небольшой, имеет смысл уменьшить размерности скрытого состояния и эмбедингов.

Также будем умнее обрабатывать падинги и брать выход сразу после последнего символа реальной строки.

In [49]:
pad_id = tokenizer.token_to_id(pad_token)
from torch.nn.utils.rnn import pack_padded_sequence

def make_rnn_class_pack_padded(arc : nn.Module):
    class RNN(nn.Module):
        def __init__(self, num_layers=1, embedding_dim=128, hidden_dim=64):
            super().__init__()

            self.emb = nn.Embedding(
                num_embeddings=tokenizer.get_vocab_size(),
                embedding_dim=embedding_dim
            )

            self.rnn = arc(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True
            )

            self.fc = nn.Linear(
                in_features=hidden_dim,
                out_features=N_CLASSES
            )

        def forward(self, batch):
            with torch.no_grad():
                lengths = (batch != pad_id).sum(dim=1).cpu()

            emb = self.emb(batch)

            packed = pack_padded_sequence(
                emb,
                lengths,
                batch_first=True,
                enforce_sorted=False
            )

            _, state = self.rnn(packed)
            h_n = state[0] if isinstance(state, tuple) else state
            return self.fc(h_n[-1])
    return RNN

In [50]:
model_params = {
    'embedding_dim' : 32,
    'hidden_dim' : 32
}

In [51]:
trained_gru, history_gru = train_loop(make_rnn_class_pack_padded(nn.GRU), model_param=model_params)

Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

 Эпоха 1/10, train_acc 0.7011833190917969, train_loss 0.7146259546279907, test_acc 0.8471052646636963, test_loss 0.41965439915657043
 Эпоха 2/10, train_acc 0.8847000002861023, train_loss 0.3318307399749756, test_acc 0.8815789222717285, test_loss 0.3344726860523224
 Эпоха 3/10, train_acc 0.9146166443824768, train_loss 0.2532740831375122, test_acc 0.8968421220779419, test_loss 0.2973153591156006
 Эпоха 4/10, train_acc 0.9306416511535645, train_loss 0.2087099701166153, test_acc 0.9039473533630371, test_loss 0.2829471528530121
 Эпоха 5/10, train_acc 0.943191647529602, train_loss 0.17531028389930725, test_acc 0.9080263376235962, test_loss 0.2829643785953522
 Эпоха 6/10, train_acc 0.9521416425704956, train_loss 0.14810964465141296, test_acc 0.9106578826904297, test_loss 0.27894267439842224
 Эпоха 7/10, train_acc 0.9603333473205566, train_loss 0.12512114644050598, test_acc 0.9084210395812988, test_loss 0.29940730333328247
 Эпоха 8/10, train_acc 0.9669749736785889, train_loss 0.107025615870952

In [52]:
trained_gru, history_gru = train_loop(
    make_rnn_class_pack_padded(nn.GRU)
    , model_param={
        'embedding_dim' : 32,
        'hidden_dim' : 32,
        'num_layers' : 2
    }
)

Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

 Эпоха 1/10, train_acc 0.7261666655540466, train_loss 0.6619248986244202, test_acc 0.8540789484977722, test_loss 0.4068956971168518
 Эпоха 2/10, train_acc 0.8932416439056396, train_loss 0.30770888924598694, test_acc 0.8894736766815186, test_loss 0.3098931908607483
 Эпоха 3/10, train_acc 0.9207333326339722, train_loss 0.23283571004867554, test_acc 0.902236819267273, test_loss 0.29169899225234985
 Эпоха 4/10, train_acc 0.9366250038146973, train_loss 0.18979831039905548, test_acc 0.9048684239387512, test_loss 0.2796213924884796
 Эпоха 5/10, train_acc 0.9484999775886536, train_loss 0.156452476978302, test_acc 0.9047368168830872, test_loss 0.29111814498901367
 Эпоха 6/10, train_acc 0.9580749869346619, train_loss 0.12986129522323608, test_acc 0.9080263376235962, test_loss 0.29801246523857117
 Эпоха 7/10, train_acc 0.9662083387374878, train_loss 0.10666058212518692, test_acc 0.9088158011436462, test_loss 0.3084831237792969
 Эпоха 8/10, train_acc 0.9729833602905273, train_loss 0.08742839097976

In [53]:
trained_gru, history_gru = train_loop(
    make_rnn_class_pack_padded(nn.GRU)
    , model_param={
        'embedding_dim' : 32,
        'hidden_dim' : 32,
        'num_layers' : 3
    }
)

Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

 Эпоха 1/10, train_acc 0.7244666814804077, train_loss 0.6612972021102905, test_acc 0.8606578707695007, test_loss 0.389381468296051
 Эпоха 2/10, train_acc 0.8937583565711975, train_loss 0.3066031336784363, test_acc 0.8947368264198303, test_loss 0.3091995120048523
 Эпоха 3/10, train_acc 0.9204166531562805, train_loss 0.2318468689918518, test_acc 0.9046052694320679, test_loss 0.2811935842037201
 Эпоха 4/10, train_acc 0.9370416402816772, train_loss 0.18692216277122498, test_acc 0.9107894897460938, test_loss 0.27054914832115173
 Эпоха 5/10, train_acc 0.9490249752998352, train_loss 0.15389351546764374, test_acc 0.9105263352394104, test_loss 0.27246204018592834
 Эпоха 6/10, train_acc 0.9587249755859375, train_loss 0.1274561882019043, test_acc 0.9065789580345154, test_loss 0.292047917842865
 Эпоха 7/10, train_acc 0.9666083455085754, train_loss 0.10407453775405884, test_acc 0.9115789532661438, test_loss 0.2985135614871979
 Эпоха 8/10, train_acc 0.9737416505813599, train_loss 0.08510946482419968

In [57]:
LOG.add_log(0.937, 0.91,'BPE', 'GRU_tuned', '32 emb и hidden, 3 слоя')
LOG.display()

,Модель,Токенизатор,train acc,test acc,замечания
0,TF-IDF + логрег,word-level,0.940,0.918,
1,самый частый класс,нет,0.250,0.250,
2,TF-IDF + логрег,nltk,0.945,0.918,
3,GRU,BPE,0.910,0.906,early stop 5 эпох
4,LSTM,BPE,0.944,0.903,early stop 7 эпох
5,RNN,BPE,0.250,0.250,"не обучалась, видимо не текли градиенты"
6,GRU_tuned,BPE,0.937,0.910,"32 emb и hidden, 3 слоя"


Сделаем дропаут на и увеличим размерность эмбедингов до 64

In [58]:
def make_rnn_class_dropout(arc : nn.Module):
    class RNN(nn.Module):
        def __init__(self, num_layers=1, embedding_dim=128, hidden_dim=64):
            super().__init__()

            self.emb = nn.Embedding(
                num_embeddings=tokenizer.get_vocab_size(),
                embedding_dim=embedding_dim
            )

            self.rnn = arc(
                input_size=embedding_dim,
                hidden_size=hidden_dim,
                num_layers=num_layers,
                batch_first=True,
                dropout=0.2
            )

            self.fc = nn.Linear(
                in_features=hidden_dim,
                out_features=N_CLASSES
            )

        def forward(self, batch):
            with torch.no_grad():
                lengths = (batch != pad_id).sum(dim=1).cpu()

            emb = self.emb(batch)

            packed = pack_padded_sequence(
                emb,
                lengths,
                batch_first=True,
                enforce_sorted=False
            )

            _, state = self.rnn(packed)
            h_n = state[0] if isinstance(state, tuple) else state
            return self.fc(h_n[-1])
    return RNN

In [59]:
trained_gru, history_gru = train_loop(
    make_rnn_class_dropout(nn.GRU),
    model_param = {
        'embedding_dim': 64,
        'hidden_dim': 32,
        'num_layers': 2
    }
)

Epochs:   0%|          | 0/10 [00:00<?, ?it/s]

 Эпоха 1/10, train_acc 0.7660166621208191, train_loss 0.5874583721160889, test_acc 0.8782894611358643, test_loss 0.34273388981819153
 Эпоха 2/10, train_acc 0.9066500067710876, train_loss 0.2719208896160126, test_acc 0.9028947353363037, test_loss 0.27820929884910583
 Эпоха 3/10, train_acc 0.9309916496276855, train_loss 0.203996479511261, test_acc 0.9130263328552246, test_loss 0.2618551254272461
 Эпоха 4/10, train_acc 0.9462249875068665, train_loss 0.1607692390680313, test_acc 0.9126315712928772, test_loss 0.2638085186481476
 Эпоха 5/10, train_acc 0.9572916626930237, train_loss 0.12797850370407104, test_acc 0.9131578803062439, test_loss 0.27925822138786316
 Эпоха 6/10, train_acc 0.9681416749954224, train_loss 0.09961557388305664, test_acc 0.910921037197113, test_loss 0.3060424029827118
 Эпоха 7/10, train_acc 0.9746500253677368, train_loss 0.07873550057411194, test_acc 0.9092105031013489, test_loss 0.33141380548477173


In [62]:
LOG.add_log(0.93, 0.913,'BPE', 'GRU + dropout', '64 emb 32 hidden, 2 слоя')
LOG.display()

,Модель,Токенизатор,train acc,test acc,замечания
0,TF-IDF + логрег,word-level,0.940,0.918,
1,самый частый класс,нет,0.250,0.250,
2,TF-IDF + логрег,nltk,0.945,0.918,
3,GRU,BPE,0.910,0.906,early stop 5 эпох
4,LSTM,BPE,0.944,0.903,early stop 7 эпох
5,RNN,BPE,0.250,0.250,"не обучалась, видимо не текли градиенты"
6,GRU_tuned,BPE,0.937,0.910,"32 emb и hidden, 3 слоя"
7,GRU + dropout,BPE,0.930,0.913,"64 emb 32 hidden, 2 слоя"


Уже сейчас можно сделать вывод что с практической точки зрения TF-IDF + логрег оказалась лучшей моделью. Мы даже не пытались тюнить логрег, а уже получили результат выше чем у нейросетей.